# Stage 2 — Data Standardization and Comparability Preparation

This notebook converts the committed source-native Top Brand and Brand Footprint evidence into reproducible long-form observations. It preserves source semantics, explicit missingness, ownership scope, category and brand mappings, and comparability restrictions. It does not calculate portfolio winners or performance scores.


## Environment Setup

Import the standard libraries used for file validation, tabular transformation, output checksums, and packaging.


In [3]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

REPOSITORY_HEAD = "b95b9b585c84c31ed78dfe0e9fbca995501fb54f"
REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
REQUIRED_FILES = ['metadata/ownership_registry.csv', 'metadata/performance_acquisition_universe.csv', 'metadata/source_registry.csv', 'metadata/performance_source_inventory.csv', 'metadata/standardization_schema.csv', 'metadata/category_crosswalk.csv', 'metadata/brand_label_crosswalk.csv', 'metadata/comparability_rules.csv', 'metadata/stage2_mapping_validation.csv', 'data/source_native/top_brand_targeted_observations.csv', 'data/source_native/brand_footprint_public_facts.csv']
EXPECTED_SHA256 = {'metadata/ownership_registry.csv': 'a30edb56b67f77857dfdf2332f3b19409db65a7d3771f7edb3af316b30b34744', 'metadata/performance_acquisition_universe.csv': '17a9126dfa46db43be0b50f4c2f30ea4ee23713526372e05697e66250d42dfb9', 'metadata/source_registry.csv': 'cb19e2efbd46eddebe7d3a22bd1ced13841920679709d93775302bc6ad76594e', 'metadata/performance_source_inventory.csv': '3e7d52d0389fb655c169faa9bda593b07c45697e594115f206b56913fd92d554', 'metadata/standardization_schema.csv': 'e29f9c7648d6d0fa821f2acf9626f7248cf3d381b5d50e34ed37e6842d602159', 'metadata/category_crosswalk.csv': '180632c43e9cb0e520f25717f3229f5828bb1bd091af76915043c409ea978b6b', 'metadata/brand_label_crosswalk.csv': '1ca5acd5a02ebe68609b9970cbffec18cb4e275c8ee2083b9341a5fa7b998963', 'metadata/comparability_rules.csv': '2e7c0659f2d7addbe41d2028fd88452a217c6bd0b49623fca70c96cb6b466a97', 'metadata/stage2_mapping_validation.csv': '1026be2930b3555808868fe4f2a0d685d91638d6d47843f3f603dab11932cc55', 'data/source_native/top_brand_targeted_observations.csv': 'dfc7be1360cc23c82dc70b96ee11c050c196f0f3c4ceb7ec84e662f60e120764', 'data/source_native/brand_footprint_public_facts.csv': '29eb6fc0ffce401f6812148e59f502ac561f265ebbbc20fc428975720967f888'}

OUTPUT_ROOT = Path(os.environ.get("FMCG_STAGE2B_OUTPUT_ROOT", "/content/stage2b_outputs"))
STANDARDIZED_ROOT = OUTPUT_ROOT / "data" / "standardized"
METADATA_ROOT = OUTPUT_ROOT / "metadata"
STANDARDIZED_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)


## Committed Input Retrieval

Load the exact Stage 2B inputs from the private GitHub repository at the locked commit. The Colab Secret named GITHUB_TOKEN is used only in the authorization header and is never printed, embedded in a URL, or written to a file.


In [4]:
# source_marker: colab_private_repository_input_retrieval
configured_root = os.environ.get("FMCG_STAGE2B_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run this notebook in Google Colab or set FMCG_STAGE2B_INPUT_ROOT for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("The Colab Secret GITHUB_TOKEN is unavailable or access has not been granted to this notebook.")

    INPUT_ROOT = Path("/content/fmcg_stage2b_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in REQUIRED_FILES:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        api_url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}/contents/"
            f"{encoded_path}?ref={REPOSITORY_HEAD}"
        )
        request = urllib.request.Request(
            api_url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage2b-colab",
            },
        )
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing_inputs = [
    relative_path
    for relative_path in REQUIRED_FILES
    if not (INPUT_ROOT / relative_path).exists()
]
if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(REQUIRED_FILES)}/{len(REQUIRED_FILES)}")


Input mode: locked_github_commit
Required files found: 11/11


## Input Integrity Validation

Verify every input against its expected SHA-256 checksum before loading any analytical registry or source-native observation.


In [5]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


checksum_rows = []
for relative_path in REQUIRED_FILES:
    actual = sha256_file(INPUT_ROOT / relative_path)
    expected = EXPECTED_SHA256[relative_path]
    checksum_rows.append(
        {
            "file_path": relative_path,
            "expected_sha256": expected,
            "actual_sha256": actual,
            "status": "passed" if actual == expected else "failed",
        }
    )

checksum_validation = pd.DataFrame(checksum_rows)
if not checksum_validation["status"].eq("passed").all():
    raise ValueError("Input checksum validation failed.")

print(checksum_validation[["file_path", "status"]].to_string(index=False))


                                             file_path status
                       metadata/ownership_registry.csv passed
         metadata/performance_acquisition_universe.csv passed
                          metadata/source_registry.csv passed
             metadata/performance_source_inventory.csv passed
                   metadata/standardization_schema.csv passed
                       metadata/category_crosswalk.csv passed
                    metadata/brand_label_crosswalk.csv passed
                      metadata/comparability_rules.csv passed
                metadata/stage2_mapping_validation.csv passed
data/source_native/top_brand_targeted_observations.csv passed
   data/source_native/brand_footprint_public_facts.csv passed


## Registry and Source-Native Loading

Load all identifiers as strings so source labels, category identifiers, blank values, dates, and provenance fields remain unchanged.


In [6]:
csv_shape_repairs = []


def read_registry(relative_path: str) -> pd.DataFrame:
    path = INPUT_ROOT / relative_path
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        rows = []
        for line_number, row in enumerate(reader, start=2):
            if len(row) > len(header):
                if header[-1] != "notes":
                    raise ValueError(
                        f"Unexpected extra CSV fields outside a terminal notes column: {relative_path}:{line_number}"
                    )
                row = row[: len(header) - 1] + [",".join(row[len(header) - 1 :])]
                csv_shape_repairs.append(
                    {
                        "file_path": relative_path,
                        "line_number": line_number,
                        "treatment": "Joined surplus terminal fields into the notes column",
                    }
                )
            if len(row) != len(header):
                raise ValueError(
                    f"CSV field-count mismatch: {relative_path}:{line_number} has {len(row)} fields; expected {len(header)}."
                )
            rows.append(row)
    return pd.DataFrame(rows, columns=header, dtype=str).fillna("")


ownership = read_registry("metadata/ownership_registry.csv")
acquisition_universe = read_registry("metadata/performance_acquisition_universe.csv")
source_registry = read_registry("metadata/source_registry.csv")
source_inventory = read_registry("metadata/performance_source_inventory.csv")
schema_registry = read_registry("metadata/standardization_schema.csv")
category_crosswalk = read_registry("metadata/category_crosswalk.csv")
brand_crosswalk = read_registry("metadata/brand_label_crosswalk.csv")
comparability_rules = read_registry("metadata/comparability_rules.csv")
mapping_validation = read_registry("metadata/stage2_mapping_validation.csv")
top_brand_native = read_registry("data/source_native/top_brand_targeted_observations.csv")
brand_footprint_native = read_registry("data/source_native/brand_footprint_public_facts.csv")

loaded_counts = pd.DataFrame(
    [
        ("ownership_registry", len(ownership)),
        ("performance_acquisition_universe", len(acquisition_universe)),
        ("source_registry", len(source_registry)),
        ("performance_source_inventory", len(source_inventory)),
        ("standardization_schema", len(schema_registry)),
        ("category_crosswalk", len(category_crosswalk)),
        ("brand_label_crosswalk", len(brand_crosswalk)),
        ("comparability_rules", len(comparability_rules)),
        ("stage2_mapping_validation", len(mapping_validation)),
        ("top_brand_source_native", len(top_brand_native)),
        ("brand_footprint_source_native", len(brand_footprint_native)),
    ],
    columns=["table", "row_count"],
)
print(loaded_counts.to_string(index=False))


                           table  row_count
              ownership_registry        157
performance_acquisition_universe         28
                 source_registry         32
    performance_source_inventory         22
          standardization_schema         51
              category_crosswalk         43
           brand_label_crosswalk         31
             comparability_rules         18
       stage2_mapping_validation         18
         top_brand_source_native         18
   brand_footprint_source_native         10


## Standardization Helpers

Define deterministic lookups for source metadata, brand identity, category taxonomy, ownership validity, and source-native value semantics.


In [7]:
SCHEMA_FIELDS = schema_registry.sort_values("field_order", key=lambda s: s.astype(int))["field_name"].tolist()
ALLOWED_COMPARABILITY_CLASSES = set(comparability_rules["default_class"])


def blank_observation() -> dict:
    return {field: "" for field in SCHEMA_FIELDS}


def inventory_row(source_family: str, edition_or_year: str) -> pd.Series:
    matches = source_inventory[
        source_inventory["source_family"].eq(source_family)
        & source_inventory["edition_or_year"].eq(str(edition_or_year))
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one source-inventory row for {source_family} {edition_or_year}; found {len(matches)}."
        )
    return matches.iloc[0]


def source_row(source_id: str) -> pd.Series:
    matches = source_registry[source_registry["source_id"].eq(source_id)]
    if len(matches) != 1:
        raise ValueError(f"Expected one source-registry row for {source_id}; found {len(matches)}.")
    return matches.iloc[0]


def brand_mapping(universe_id: str, source_family: str, source_label: str, year: int | None, value_present: bool) -> pd.Series:
    matches = brand_crosswalk[
        brand_crosswalk["universe_id"].eq(universe_id)
        & brand_crosswalk["source_family"].eq(source_family)
        & brand_crosswalk["source_brand_label"].eq(source_label)
    ].copy()
    if matches.empty:
        raise ValueError(f"No brand mapping for {universe_id} / {source_family} / {source_label}.")

    if year is not None and len(matches) >= 1:
        valid = matches[
            matches["valid_from"].apply(lambda x: not x or year >= int(x))
            & matches["valid_to"].apply(lambda x: not x or year <= int(x))
        ]
        if len(valid) == 1:
            return valid.iloc[0]
        if value_present:
            raise ValueError(f"Observed value falls outside the mapped label period for {universe_id} / {source_label} / {year}.")

    if len(matches) != 1:
        raise ValueError(f"Ambiguous brand mapping for {universe_id} / {source_label}.")
    return matches.iloc[0]


def top_brand_category(source_subcategory_id: str) -> pd.Series:
    matches = category_crosswalk[
        category_crosswalk["input_domain"].eq("source_native_performance")
        & category_crosswalk["source_family"].eq("Top Brand")
        & category_crosswalk["source_subcategory_id"].eq(source_subcategory_id)
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Top Brand category mapping for subcategory {source_subcategory_id}.")
    return matches.iloc[0]


def ownership_category(canonical_group: str, canonical_brand_family: str) -> tuple[str, str, str, str]:
    records = ownership[
        ownership["group"].eq(canonical_group)
        & ownership["brand_family"].str.casefold().eq(canonical_brand_family.casefold())
    ]
    categories = records["category_scope"].drop_duplicates().tolist()
    if len(categories) != 1:
        return "", "", "", "requires_brand_level_mapping"

    matches = category_crosswalk[
        category_crosswalk["input_domain"].eq("ownership_registry")
        & category_crosswalk["source_category"].eq(categories[0])
    ]
    if len(matches) != 1:
        return "", "", "", "unresolved"

    mapped = matches.iloc[0]
    if not mapped["canonical_sector"]:
        return "", "", "", mapped["mapping_status"]
    return (
        mapped["canonical_sector"],
        mapped["canonical_category"],
        mapped["canonical_subcategory"],
        mapped["mapping_status"],
    )


def methodology_fields(inventory: pd.Series) -> tuple[str, str]:
    status = inventory["methodology_status"]
    if status == "current_methodology_documented":
        return "top_brand_current_documented", "documented"
    if status == "historical_methodology_not_verified":
        return "top_brand_historical_unverified", "historical_not_verified"
    if status == "national_coverage_documented":
        return "brand_footprint_national", "documented"
    if status == "urban_edition_documented":
        return "brand_footprint_urban", "documented"
    if status == "methodology_partially_documented":
        return "brand_footprint_partial", "partially_documented"
    return "unknown", "not_available"


def ownership_status(mapping: pd.Series, reference_period: str) -> str:
    if mapping["attribution_scope"] == "extended_group_sensitivity_only":
        return "valid_with_caveat"
    year_match = re.search(r"(19|20)\d{2}", str(reference_period))
    reference_year = int(year_match.group(0)) if year_match else None
    if mapping["relationship_type"] == "former_portfolio":
        if not mapping["valid_to"] or reference_year is None:
            return "unresolved"
        return "valid" if reference_year <= int(mapping["valid_to"][:4]) else "invalid"
    if mapping["relationship_type"] in {"controlled", "controlled_group_portfolio"}:
        return "valid"
    if mapping["relationship_type"] in {"related_group_affiliate", "joint_venture"}:
        return "valid_with_caveat"
    return "unresolved"


## Top Brand Wide-to-Long Standardization

Expand every 2022–2026 source cell into an explicit observation, including unavailable cells with null values rather than zeros.


In [8]:
top_brand_rows = []
value_columns = [f"value_{year}" for year in range(2022, 2027)]
universe_lookup = acquisition_universe.set_index("universe_id", drop=False)

for source_row_number, (_, native) in enumerate(top_brand_native.iterrows(), start=1):
    universe = universe_lookup.loc[native["universe_id"]]
    category = top_brand_category(native["source_subcategory_id"])

    for value_column in value_columns:
        year = int(value_column.split("_")[-1])
        raw_value = native[value_column]
        value_present = raw_value != ""
        mapping = brand_mapping(
            native["universe_id"],
            "Top Brand",
            native["source_brand_label"],
            year,
            value_present,
        )
        inventory = inventory_row("Top Brand", str(year))
        methodology_cluster, methodology_status = methodology_fields(inventory)
        source = source_row("SRC_TBA_004")

        row = blank_observation()
        row.update(
            {
                "observation_id": f"STD_{native['record_id']}_{year}",
                "source_native_file": "data/source_native/top_brand_targeted_observations.csv",
                "source_native_record_id": native["record_id"],
                "source_native_row_number": str(source_row_number),
                "source_value_column": value_column,
                "universe_id": native["universe_id"],
                "source_id": "SRC_TBA_004",
                "source_family": "Top Brand",
                "publisher": source["publisher"],
                "source_url": native["source_url"],
                "retrieval_date": native["retrieval_date"],
                "edition": "",
                "publication_year": str(year),
                "reference_period": str(year),
                "survey_phase": native["survey_phase"],
                "geography": inventory["geography"],
                "methodology_cluster": methodology_cluster,
                "methodology_status": methodology_status,
                "source_group_label": native["group"],
                "canonical_group": mapping["canonical_group"],
                "operating_company": mapping["operating_company"],
                "attribution_scope": mapping["attribution_scope"],
                "relationship_type": mapping["relationship_type"],
                "ownership_start": "",
                "ownership_end": "",
                "ownership_period_status": ownership_status(mapping, str(year)),
                "source_brand_label": native["source_brand_label"],
                "canonical_brand_family": mapping["canonical_brand_family"],
                "canonical_brand_key": mapping["canonical_brand_key"],
                "brand_identity_status": mapping["identity_status"],
                "brand_mapping_status": mapping["mapping_status"],
                "source_category": universe["source_category"],
                "source_subcategory": native["source_subcategory"],
                "source_subcategory_id": native["source_subcategory_id"],
                "canonical_sector": category["canonical_sector"],
                "canonical_category": category["canonical_category"],
                "canonical_subcategory": category["canonical_subcategory"],
                "category_mapping_status": category["mapping_status"],
                "metric": native["metric"],
                "value": raw_value,
                "value_text": "",
                "unit": native["unit"],
                "observation_status": "observed" if value_present else "not_available",
                "missingness_reason": "" if value_present else "No value is exposed in the source-native comparison row for this year.",
                "value_semantics": "exact_numeric" if value_present else "unavailable",
                "bound_type": "none" if value_present else "not_applicable",
                "uncertainty_status": "not_reported" if value_present else "not_applicable",
                "comparability_class": "comparable_with_caveat" if value_present else "not_comparable",
                "comparability_rule_id": "CMP002" if value_present else "CMP004",
                "comparability_reason": (
                    "Same brand and source subcategory; historical methodology and survey-surface caveats remain."
                    if value_present
                    else "The source-native year cell is unavailable and is retained without imputation."
                ),
                "notes": native["notes"],
            }
        )
        top_brand_rows.append(row)

top_brand_standardized = pd.DataFrame(top_brand_rows, columns=SCHEMA_FIELDS)
print(
    top_brand_standardized.groupby("observation_status", dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
    .to_string(index=False)
)


observation_status  row_count
     not_available         10
          observed         80


## Brand Footprint Fact Standardization

Retain each limited public fact as one observation while preserving edition, reference period, geography, bounds, qualitative semantics, and attribution scope.


In [9]:
qualitative_values = {
    "growth_status": "one_of_fastest_growing_top_20_brands",
    "selection_status": "selected_by_indonesian_consumers",
}
metric_rules = {
    "brand_rank": ("CMP005", "comparable_with_caveat", "ordinal_rank", "none"),
    "growth_status": ("CMP006", "context_only", "qualitative_status", "not_applicable"),
    "selection_status": ("CMP007", "context_only", "qualitative_status", "not_applicable"),
    "household_reach_lower_bound": ("CMP008", "context_only", "lower_bound", "lower_bound"),
    "top100_entry": ("CMP009", "context_only", "boolean_indicator", "none"),
    "rank_change": ("CMP010", "context_only", "exact_numeric", "none"),
}

brand_footprint_rows = []
for source_row_number, (_, native) in enumerate(brand_footprint_native.iterrows(), start=1):
    mapping = brand_mapping(
        native["universe_id"],
        "Brand Footprint",
        native["source_brand_label"],
        int(native["edition"]),
        native["value"] != "",
    )
    inventory = inventory_row("Brand Footprint", native["edition"])
    methodology_cluster, methodology_status = methodology_fields(inventory)
    source_matches = source_registry[source_registry["source_url"].eq(native["source_url"])]
    if len(source_matches) != 1:
        raise ValueError(f"Expected one source-registry URL match for {native['fact_id']}.")
    source = source_matches.iloc[0]
    sector, category_name, subcategory_name, category_status = ownership_category(
        mapping["canonical_group"],
        mapping["canonical_brand_family"],
    )
    rule_id, comparison_class, value_semantics, bound_type = metric_rules[native["metric"]]

    row = blank_observation()
    row.update(
        {
            "observation_id": f"STD_{native['fact_id']}",
            "source_native_file": "data/source_native/brand_footprint_public_facts.csv",
            "source_native_record_id": native["fact_id"],
            "source_native_row_number": str(source_row_number),
            "source_value_column": "value",
            "universe_id": native["universe_id"],
            "source_id": source["source_id"],
            "source_family": native["source_family"],
            "publisher": source["publisher"],
            "source_url": native["source_url"],
            "retrieval_date": native["retrieval_date"],
            "edition": native["edition"],
            "publication_year": native["edition"],
            "reference_period": native["reference_period"],
            "survey_phase": "",
            "geography": native["geography"],
            "methodology_cluster": methodology_cluster,
            "methodology_status": methodology_status,
            "source_group_label": native["group"],
            "canonical_group": mapping["canonical_group"],
            "operating_company": mapping["operating_company"],
            "attribution_scope": mapping["attribution_scope"],
            "relationship_type": mapping["relationship_type"],
            "ownership_start": "",
            "ownership_end": "",
            "ownership_period_status": ownership_status(mapping, native["reference_period"]),
            "source_brand_label": native["source_brand_label"],
            "canonical_brand_family": mapping["canonical_brand_family"],
            "canonical_brand_key": mapping["canonical_brand_key"],
            "brand_identity_status": mapping["identity_status"],
            "brand_mapping_status": mapping["mapping_status"],
            "source_category": "National urban+rural",
            "source_subcategory": "public_summary_facts",
            "source_subcategory_id": "",
            "canonical_sector": sector,
            "canonical_category": category_name,
            "canonical_subcategory": subcategory_name,
            "category_mapping_status": category_status,
            "metric": native["metric"],
            "value": native["value"],
            "value_text": qualitative_values.get(native["metric"], ""),
            "unit": native["unit"],
            "observation_status": native["observation_status"],
            "missingness_reason": "",
            "value_semantics": value_semantics,
            "bound_type": bound_type,
            "uncertainty_status": (
                "qualitative_caveat"
                if comparison_class == "context_only" or bound_type == "lower_bound"
                else "not_reported"
            ),
            "comparability_class": comparison_class,
            "comparability_rule_id": rule_id,
            "comparability_reason": comparability_rules.loc[
                comparability_rules["rule_id"].eq(rule_id), "notes"
            ].iloc[0],
            "notes": native["notes"],
        }
    )
    brand_footprint_rows.append(row)

brand_footprint_standardized = pd.DataFrame(brand_footprint_rows, columns=SCHEMA_FIELDS)
print(
    brand_footprint_standardized[
        ["observation_id", "canonical_group", "canonical_brand_family", "metric", "comparability_class", "attribution_scope"]
    ].to_string(index=False)
)


observation_id    canonical_group canonical_brand_family                      metric    comparability_class               attribution_scope
    STD_BFP001           Indofood                Indomie                  brand_rank comparable_with_caveat                  strict_control
    STD_BFP002 Unilever Indonesia               Lifebuoy                  brand_rank comparable_with_caveat                  strict_control
    STD_BFP003           Indofood               Indofood                  brand_rank comparable_with_caveat                  strict_control
    STD_BFP004        Wings Group                Ekonomi               growth_status           context_only                  strict_control
    STD_BFP005 Unilever Indonesia                  Royco            selection_status           context_only                  strict_control
    STD_BFP006 Unilever Indonesia                  Bango            selection_status           context_only                  strict_control
    STD_BFP007      

## Canonical Observation Assembly

Combine the source-specific standardized tables without pooling metrics, units, ranks, bounds, or qualitative facts into a shared performance scale.


In [10]:
performance_observations = pd.concat(
    [top_brand_standardized, brand_footprint_standardized],
    ignore_index=True,
)[SCHEMA_FIELDS]

summary = (
    performance_observations.groupby(
        ["source_family", "metric", "unit", "comparability_class"],
        dropna=False,
    )
    .size()
    .rename("row_count")
    .reset_index()
)
print(summary.to_string(index=False))


  source_family                      metric                 unit    comparability_class  row_count
Brand Footprint                  brand_rank                 rank comparable_with_caveat          3
Brand Footprint               growth_status          text_status           context_only          1
Brand Footprint household_reach_lower_bound   percent_households           context_only          1
Brand Footprint                 rank_change            positions           context_only          1
Brand Footprint            selection_status          text_status           context_only          2
Brand Footprint                top100_entry              boolean           context_only          2
      Top Brand                         TBI percent_index_points comparable_with_caveat         80
      Top Brand                         TBI percent_index_points         not_comparable         10


## Stage 2B Validation

Validate row preservation, explicit missingness, mapping completeness, ownership scope, metric semantics, and output uniqueness before files are written.


In [11]:
validation_rows = []


def add_check(check_id: str, area: str, description: str, condition: bool, result: str, critical: bool, treatment: str, notes: str = "") -> None:
    validation_rows.append(
        {
            "check_id": check_id,
            "validation_area": area,
            "check_description": description,
            "result": result,
            "status": "passed" if condition else ("failed" if critical else "passed_with_caveat"),
            "critical_failure": "no" if condition else ("yes" if critical else "no"),
            "required_treatment": treatment,
            "notes": notes,
        }
    )


add_check("S2B001", "input_integrity", "All checksum-locked inputs match the expected Stage 2B bundle", checksum_validation["status"].eq("passed").all(), f"{checksum_validation['status'].eq('passed').sum()}/{len(checksum_validation)} passed", True, "Stop transformation when an input checksum differs")
add_check("S2B002", "top_brand_preservation", "All source-native Top Brand rows are retained", top_brand_standardized["source_native_record_id"].nunique() == len(top_brand_native), f"{top_brand_standardized['source_native_record_id'].nunique()}/{len(top_brand_native)} source rows retained", True, "Retain every source-native record ID")
add_check("S2B003", "top_brand_expansion", "Each Top Brand row expands to five explicit year cells", len(top_brand_standardized) == len(top_brand_native) * 5, f"{len(top_brand_standardized)} long rows", True, "Emit one row per 2022–2026 source cell")
add_check("S2B004", "top_brand_observed", "Observed Top Brand values retain the Stage 1 count", top_brand_standardized["observation_status"].eq("observed").sum() == 80, f"{top_brand_standardized['observation_status'].eq('observed').sum()} observed values", True, "Preserve every observed TBI value")
add_check("S2B005", "missingness", "Unavailable Top Brand cells remain explicit null observations", top_brand_standardized["observation_status"].eq("not_available").sum() == 10, f"{top_brand_standardized['observation_status'].eq('not_available').sum()} unavailable cells", True, "Do not replace missing values with zero")
add_check("S2B006", "missingness", "Unavailable observations contain no numeric value", top_brand_standardized.loc[top_brand_standardized["observation_status"].eq("not_available"), "value"].eq("").all(), "All unavailable values are blank", True, "Keep unavailable values null")
add_check("S2B007", "brand_footprint_preservation", "Every limited Brand Footprint fact is retained once", len(brand_footprint_standardized) == len(brand_footprint_native) == 10, f"{len(brand_footprint_standardized)} facts retained", True, "Retain each source-native fact exactly once")
add_check("S2B008", "output_completeness", "Canonical performance table contains the expected observations", len(performance_observations) == 100, f"{len(performance_observations)} canonical rows", True, "Require 90 Top Brand rows and 10 Brand Footprint rows")
add_check("S2B009", "uniqueness", "Canonical observation identifiers are unique", performance_observations["observation_id"].is_unique, f"{performance_observations['observation_id'].nunique()} unique IDs", True, "Resolve duplicate observation IDs")
add_check("S2B010", "schema", "Canonical outputs follow the committed field order", performance_observations.columns.tolist() == SCHEMA_FIELDS, f"{len(SCHEMA_FIELDS)} fields in canonical order", True, "Use the committed standardization schema")
add_check("S2B011", "brand_mapping", "All standardized observations have canonical brand mappings", performance_observations["canonical_brand_key"].ne("").all(), f"{performance_observations['canonical_brand_key'].ne('').sum()}/{len(performance_observations)} mapped", True, "Do not standardize unmapped brand identities")
add_check("S2B012", "category_mapping", "Every Top Brand observation has a category-specific taxonomy mapping", top_brand_standardized["canonical_sector"].ne("").all(), f"{top_brand_standardized['canonical_sector'].ne('').sum()}/{len(top_brand_standardized)} mapped", True, "Require source-subcategory category mapping")
add_check("S2B013", "comparability", "Every comparability class exists in the committed rule registry", performance_observations["comparability_class"].isin(ALLOWED_COMPARABILITY_CLASSES).all(), "All classes recognized", True, "Reject unregistered comparability classes")
add_check("S2B014", "metric_semantics", "TBI remains source-native and is never relabeled as market share", not performance_observations["metric"].str.contains("market_share", case=False, regex=False).any(), "No market-share relabeling", True, "Retain TBI as TBI")
add_check("S2B015", "ownership_scope", "Le Minerale remains extended-group sensitivity only", brand_footprint_standardized.loc[brand_footprint_standardized["canonical_brand_family"].eq("Le Minerale"), "attribution_scope"].eq("extended_group_sensitivity_only").all(), "Le Minerale sensitivity scope retained", True, "Exclude from strict-control Mayora aggregation")
add_check("S2B016", "brand_identity", "Cap Bango and BANGO source aliases remain visible", {"Cap Bango", "BANGO"}.issubset(set(top_brand_standardized["source_brand_label"])), "Both aliases retained", True, "Preserve source labels and family continuity caveat")
add_check("S2B017", "brand_identity", "Kopiko observations remain category-bound", brand_footprint_standardized.loc[brand_footprint_standardized["canonical_brand_family"].eq("Kopiko")].empty and top_brand_standardized.loc[top_brand_standardized["canonical_brand_family"].eq("Kopiko"), "canonical_subcategory"].eq("White Coffee").all(), "Captured Kopiko rows restricted to White Coffee", True, "Do not reuse across confectionery and coffee categories")
add_check("S2B018", "ownership_period", "No standardized observation is attributed outside a valid ownership period", not performance_observations["ownership_period_status"].eq("invalid").any(), "0 invalid ownership-period rows", True, "Exclude invalid temporal attribution")
lower_bound_rows = brand_footprint_standardized["metric"].eq("household_reach_lower_bound")
add_check("S2B019", "bound_semantics", "Household reach lower bounds remain bounds rather than exact estimates", brand_footprint_standardized.loc[lower_bound_rows, "bound_type"].eq("lower_bound").all(), "Lower-bound semantics retained", True, "Do not convert more-than statements into exact estimates")
add_check("S2B020", "provenance", "Every canonical observation retains source ID, URL, publisher, and retrieval date", performance_observations[["source_id", "source_url", "publisher", "retrieval_date"]].ne("").all().all(), "Complete provenance fields", True, "Do not publish observations with incomplete source provenance")
add_check("S2B021", "source_separation", "Source-native metric and unit combinations remain distinct", not performance_observations.assign(metric_unit=performance_observations["metric"] + "|" + performance_observations["unit"])["metric_unit"].eq("TBI|rank").any(), "No incompatible unit substitution", True, "Preserve source-native metric and unit")
add_check("S2B022", "structural_limitation", "Selective source coverage remains an explicit analytical caveat", True, "PASS_WITH_CAVEAT", False, "Use observable denominators and keep structural breadth ownership-based", "This validation is not a performance result.")
validation_rows[-1]["status"] = "passed_with_caveat"

expected_shape_repair = (
    len(csv_shape_repairs) == 1
    and csv_shape_repairs[0]["file_path"] == "data/source_native/brand_footprint_public_facts.csv"
    and csv_shape_repairs[0]["line_number"] == 8
)
add_check("S2B023", "source_native_structure", "Known surplus delimiter in the terminal Brand Footprint notes field is repaired deterministically", expected_shape_repair, f"{len(csv_shape_repairs)} terminal-notes repair", True, "Join only surplus terminal fields into notes and preserve the complete text", "No analytical field is inferred or altered.")
if expected_shape_repair:
    validation_rows[-1]["status"] = "passed_with_caveat"

add_check("S2B024", "stage_gate", "Stage 2B standardization is complete enough for user execution and review", True, "PASS_WITH_CAVEAT", False, "Review the executed notebook and generated output package before repository publication", "No winner, composite score, or analytical performance result is produced.")
validation_rows[-1]["status"] = "passed_with_caveat"

stage2b_validation = pd.DataFrame(validation_rows)
critical_failures = stage2b_validation["critical_failure"].eq("yes").sum()
if critical_failures:
    raise AssertionError(f"Stage 2B has {critical_failures} critical validation failure(s).")

print(stage2b_validation[["check_id", "validation_area", "result", "status"]].to_string(index=False))
print("Final gate: PASS_WITH_CAVEAT")


check_id              validation_area                                          result             status
  S2B001              input_integrity                                    11/11 passed             passed
  S2B002       top_brand_preservation                      18/18 source rows retained             passed
  S2B003          top_brand_expansion                                    90 long rows             passed
  S2B004           top_brand_observed                              80 observed values             passed
  S2B005                  missingness                            10 unavailable cells             passed
  S2B006                  missingness                All unavailable values are blank             passed
  S2B007 brand_footprint_preservation                               10 facts retained             passed
  S2B008          output_completeness                              100 canonical rows             passed
  S2B009                   uniqueness                  

## Output Writing and Manifest

Write standardized source tables, the combined canonical table, validation results, and an output manifest with reproducible checksums.


In [12]:
top_brand_path = STANDARDIZED_ROOT / "top_brand_observations_long.csv"
brand_footprint_path = STANDARDIZED_ROOT / "brand_footprint_facts_standardized.csv"
performance_path = STANDARDIZED_ROOT / "performance_observations.csv"
validation_path = METADATA_ROOT / "stage2_standardization_validation.csv"
manifest_path = METADATA_ROOT / "stage2_output_manifest.csv"

top_brand_standardized.to_csv(top_brand_path, index=False)
brand_footprint_standardized.to_csv(brand_footprint_path, index=False)
performance_observations.to_csv(performance_path, index=False)
stage2b_validation.to_csv(validation_path, index=False)

manifest_rows = []
for path, description in [
    (top_brand_path, "Long-form Top Brand observations including explicit unavailable cells"),
    (brand_footprint_path, "Standardized limited public Brand Footprint facts"),
    (performance_path, "Combined canonical performance observations without metric pooling"),
    (validation_path, "Stage 2B standardization validation results"),
]:
    manifest_rows.append(
        {
            "file_path": str(path.relative_to(OUTPUT_ROOT)),
            "row_count": str(len(pd.read_csv(path, dtype=str, keep_default_na=False))),
            "sha256": sha256_file(path),
            "description": description,
            "repository_head": REPOSITORY_HEAD,
        }
    )

stage2_output_manifest = pd.DataFrame(manifest_rows)
stage2_output_manifest.to_csv(manifest_path, index=False)
print(stage2_output_manifest.to_string(index=False))


                                               file_path row_count                                                           sha256                                                           description                          repository_head
       data/standardized/top_brand_observations_long.csv        90 2a3ff25291d96efd8482fb8482c15e0e8343cd101d824b55176795aea83c2516 Long-form Top Brand observations including explicit unavailable cells b95b9b585c84c31ed78dfe0e9fbca995501fb54f
data/standardized/brand_footprint_facts_standardized.csv        10 72b9038a967a72ced98aadb5b5e6268e268d222011fc85e6324873a7e2081d8a                     Standardized limited public Brand Footprint facts b95b9b585c84c31ed78dfe0e9fbca995501fb54f
          data/standardized/performance_observations.csv       100 63b30fca6260ab12acc1a15ec7d96acd945c3a2d80de1416f3ff6cd9d0a9cc53    Combined canonical performance observations without metric pooling b95b9b585c84c31ed78dfe0e9fbca995501fb54f
          metadata/stage2_st

## Colab Output Package

Package the validated Stage 2B outputs for review before repository publication.


In [13]:
OUTPUT_ZIP = OUTPUT_ROOT / "FMCG_Stage2B_Outputs.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*.csv")):
        archive.write(path, path.relative_to(OUTPUT_ROOT))

print(f"Output package: {OUTPUT_ZIP}")
print(f"Package SHA-256: {sha256_file(OUTPUT_ZIP)}")

if "google.colab" in globals().get("sys", __import__("sys")).modules and not os.environ.get("FMCG_STAGE2B_LOCAL_QA"):
    from google.colab import files as colab_files
    colab_files.download(str(OUTPUT_ZIP))


Output package: /content/stage2b_outputs/FMCG_Stage2B_Outputs.zip
Package SHA-256: 6c248f94b43294a7365906b956ac90cfea9d9784b13445359f7572f0754ce7b6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>